In [2]:
import pandas as pd
import numpy as np
import os
import json
import ipywidgets as widgets
from IPython.display import display

In [3]:
player_input = widgets.Text(
    placeholder='Enter player name',
    description='Player:'
)

display(player_input)

Text(value='', description='Player:', placeholder='Enter player name')

In [20]:
if not player_input.value.strip():
    raise ValueError("Please enter a player name in the previous cell before proceeding.")

player_name = player_input.value.strip()

In [21]:
combined_data_shots = pd.read_excel(f'../../data/mens/{player_name}/combined.xlsx', sheet_name='Shots')
combined_data_points = pd.read_excel(f'../../data/mens/{player_name}/combined.xlsx', sheet_name='Points')
combined_data_games = pd.read_excel(f'../../data/mens/{player_name}/combined.xlsx', sheet_name='Games')
combined_data_sets = pd.read_excel(f'../../data/mens/{player_name}/combined.xlsx', sheet_name='Sets')
combined_data_stats = pd.read_excel(f'../../data/mens/{player_name}/combined.xlsx', sheet_name='Stats')

Helper Functions

In [22]:
def scale_coords(df):
    df = df.copy()
    df["hit_x"]    = df["Hit (x)"]
    df["hit_y"]    = df["Hit (y)"] 
    df["bounce_x"] = df["Bounce (x)"]
    df["bounce_y"] = df["Bounce (y)"]
    return df
 
 
def classify_hit_side(hit_x):
    if pd.isna(hit_x):
        return np.nan
    return "Backhand Corner" if hit_x < 0 else np.nan
 
 
def classify_ball_direction(hit_x, bounce_x):
    if pd.isna(hit_x) or pd.isna(bounce_x):
        return np.nan
    return "Cross Court" if (hit_x * bounce_x < 0) else "Down the Line"
 
 
def classify_depth_label(result, bounce_depth):
    result_clean = str(result).strip().lower()
    if result_clean == "in":
        depth = str(bounce_depth).strip().lower()
        return {"deep": "Offensive", "short": "Defensive"}.get(depth, "Defensive")
    elif result_clean == "net":
        return "Missed - Net"
    else:
        return "Missed - Out"
 
 
def classify_miss_quality(outcome, depth_label):
    if outcome == "Miss":
        
        return "Good Miss" if depth_label == "Offensive" else "Bad Miss"
    return outcome
 
 
def compute_shot_vector(df):
    df = df.copy()
    df["vec_x"]         = df["bounce_x"] - df["hit_x"]
    df["vec_y"]         = df["bounce_y"] - df["hit_y"]
    df["vec_magnitude"] = np.sqrt(df["vec_x"] ** 2 + df["vec_y"] ** 2)
    df["vec_angle_deg"] = np.degrees(np.arctan2(df["vec_x"], df["vec_y"]))
    df["x_deviation"]   = df["bounce_x"].abs()
    return df
 
 
def get_opponent_positions(df_all_shots, io_fh_df, opp_name):
    opp = df_all_shots[df_all_shots["Player"] == opp_name][
        ["Point", "Game", "Set", "__source_file__", "Shot", "Hit (x)", "Hit (y)"]
    ].copy()
    opp = opp.rename(columns={"Shot": "opp_shot", "Hit (x)": "opp_hit_x_raw", "Hit (y)": "opp_hit_y_raw"})
    opp["opp_hit_x"] = opp["opp_hit_x_raw"] 
    opp["opp_hit_y"] = opp["opp_hit_y_raw"]
 
    rows = []
    for _, row in io_fh_df.iterrows():
        mask = (
            (opp["Point"] == row["Point"]) &
            (opp["Game"]  == row["Game"])  &
            (opp["Set"]   == row["Set"])   &
            (opp["__source_file__"] == row["__source_file__"]) &
            (opp["opp_shot"] < row["Shot"])
        )
        candidates = opp[mask]
        if not candidates.empty:
            last = candidates.sort_values("opp_shot").iloc[-1]
            rows.append({"Point": row["Point"], "Game": row["Game"], "Set": row["Set"],
                         "__source_file__": row["__source_file__"], "Shot": row["Shot"],
                         "opp_hit_x": last["opp_hit_x"], "opp_hit_y": last["opp_hit_y"]})
        else:
            rows.append({"Point": row["Point"], "Game": row["Game"], "Set": row["Set"],
                         "__source_file__": row["__source_file__"], "Shot": row["Shot"],
                         "opp_hit_x": np.nan, "opp_hit_y": np.nan})
 
    opp_pos = pd.DataFrame(rows)
    return pd.merge(io_fh_df, opp_pos, on=["Point", "Game", "Set", "__source_file__", "Shot"], how="left")

In [23]:
combined = pd.merge(
    combined_data_shots,
    combined_data_points[["Point", "Game", "Set", "Point Winner", "Match Server", "__source_file__"]],
    on=["Point", "Game", "Set", "__source_file__"],
    how="left"
)
 
io_fh = combined[
    (combined["Player"]    == player_name) &
    (combined["Stroke"]    == "Forehand") &
    (combined["Direction"] == "inside out")
].copy()
 
print(f"Total inside-out forehand shots: {len(io_fh)}")

Total inside-out forehand shots: 46


In [24]:
io_fh = scale_coords(io_fh)
 
io_fh["hit_side_zone"]  = io_fh["hit_x"].apply(classify_hit_side)
io_fh["ball_direction"] = io_fh.apply(lambda r: classify_ball_direction(r["hit_x"], r["bounce_x"]), axis=1)
io_fh["depth_label"]    = io_fh.apply(lambda r: classify_depth_label(r["Result"], r.get("Bounce Depth")), axis=1)
io_fh["outcome"]        = io_fh.apply(
    lambda r: "Won" if str(r["Result"]).strip().lower() == "in" and r["Point Winner"] == player_name
    else ("Neutral" if str(r["Result"]).strip().lower() == "in" else "Miss"), axis=1
)
io_fh["miss_quality"] = io_fh.apply(lambda r: classify_miss_quality(r["outcome"], r["depth_label"]), axis=1)

In [25]:
io_fh_bh = io_fh[io_fh["hit_side_zone"] == "Backhand Corner"].copy()
io_fh_fh = io_fh[io_fh["hit_side_zone"] == "Forehand Corner"].copy()
 
print(f"  From Backhand Corner (true IO-FH): {len(io_fh_bh)}")
print(f"  From Forehand Corner:              {len(io_fh_fh)}")

  From Backhand Corner (true IO-FH): 22
  From Forehand Corner:              0


In [26]:
all_players = combined_data_shots["Player"].dropna().unique()
opponents   = [p for p in all_players if p != player_name]
print(f"Opponents found: {opponents}")
 
if opponents:
    opponent_name = opponents[0]
    io_fh    = get_opponent_positions(combined_data_shots, io_fh, opponent_name)
    io_fh_bh = io_fh[io_fh["hit_side_zone"] == "Backhand Corner"].copy()
    io_fh_fh = io_fh[io_fh["hit_side_zone"] == "Forehand Corner"].copy()
    print(f"Opponent positions added using: {opponent_name}")
else:
    print("Warning: No opponent found — skipping opponent position step.")

Opponents found: ['Alejandro Arcila', 'Stian Klaasen', 'Matias Ponce de Leon', 'Gustavo Ribeiero']
Opponent positions added using: Alejandro Arcila


In [27]:
io_fh    = compute_shot_vector(io_fh)
io_fh_bh = io_fh[io_fh["hit_side_zone"] == "Backhand Corner"].copy()
io_fh_fh = io_fh[io_fh["hit_side_zone"] == "Forehand Corner"].copy()

In [28]:
def print_summary(df, label):
    total = len(df)
    if total == 0:
        print(f"\n{label}: No shots found.")
        return
    in_s  = df[df["Result"].str.strip().str.lower() == "in"]
    made  = df[df["miss_quality"].isin(["Won", "Neutral"])]
    won   = df[df["miss_quality"] == "Won"]
 
    print(f"\n{'='*52}")
    print(f"  {label}  (n={total})")
    print(f"{'='*52}")
    for k, v in df["miss_quality"].value_counts().items():
        print(f"  {k:<22} {v:>4}  ({round(v/total*100,1)}%)")
    print(f"\n  Depth of {len(in_s)} shots in:")
    for k, v in in_s["depth_label"].value_counts().items():
        pct = round(v / len(in_s) * 100, 1) if len(in_s) > 0 else 0
        print(f"  {k:<22} {v:>4}  ({pct}%)")
    print(f"\n  Ball Direction:")
    for k, v in df["ball_direction"].value_counts().items():
        print(f"  {k:<22} {v:>4}  ({round(v/total*100,1)}%)")
    win_rate = round(len(won) / len(made) * 100, 1) if len(made) > 0 else 0
    print(f"\n  Point win rate (shots made): {win_rate}%")
 
print_summary(io_fh,    "All Inside-Out Forehands")
print_summary(io_fh_bh, "From Backhand Corner (True IO-FH)")
print_summary(io_fh_fh, "From Forehand Corner")


  All Inside-Out Forehands  (n=46)
  Neutral                  41  (89.1%)
  Bad Miss                  5  (10.9%)

  Depth of 41 shots in:
  Offensive                25  (61.0%)
  Defensive                16  (39.0%)

  Ball Direction:
  Cross Court              42  (91.3%)
  Down the Line             4  (8.7%)

  Point win rate (shots made): 0.0%

  From Backhand Corner (True IO-FH)  (n=22)
  Neutral                  21  (95.5%)
  Bad Miss                  1  (4.5%)

  Depth of 21 shots in:
  Offensive                16  (76.2%)
  Defensive                 5  (23.8%)

  Ball Direction:
  Cross Court              21  (95.5%)
  Down the Line             1  (4.5%)

  Point win rate (shots made): 0.0%

From Forehand Corner: No shots found.


In [29]:
def print_hit_location(df, label):
    if len(df) == 0:
        return
    total = len(df)
    behind = df[df["hit_y"] < 0]
    inside = df[df["hit_y"] >= 0]
    print(f"\n{label} — Hit Location:")
    print(f"  hit_x  mean={df['hit_x'].mean():.1f}  median={df['hit_x'].median():.1f}  "
          f"std={df['hit_x'].std():.1f}  range=[{df['hit_x'].min():.1f}, {df['hit_x'].max():.1f}]")
    print(f"  hit_y  mean={df['hit_y'].mean():.1f}  median={df['hit_y'].median():.1f}  "
          f"std={df['hit_y'].std():.1f}  range=[{df['hit_y'].min():.1f}, {df['hit_y'].max():.1f}]")
    print(f"  Behind baseline (defensive): {len(behind)} ({round(len(behind)/total*100,1)}%)")
    print(f"  Inside/on baseline:          {len(inside)} ({round(len(inside)/total*100,1)}%)")
 
print_hit_location(io_fh_bh, "Backhand Corner IO-FH")

threshold_x = io_fh_bh["hit_x"].median() if len(io_fh_bh) > 0 else 0.0
print(f"\nInside-Out Forehand Zone Threshold (median hit_x): {threshold_x:.1f}")


Backhand Corner IO-FH — Hit Location:
  hit_x  mean=-1.7  median=-1.7  std=0.9  range=[-3.4, -0.2]
  hit_y  mean=-1.0  median=-1.2  std=1.3  range=[-2.7, 2.7]
  Behind baseline (defensive): 17 (77.3%)
  Inside/on baseline:          5 (22.7%)

Inside-Out Forehand Zone Threshold (median hit_x): -1.7


In [30]:
def build_records(df):
    records = []
    for _, r in df.iterrows():
        rec = {
            "pointNumber":   int(r["Point"])  if not pd.isna(r["Point"])  else None,
            "game":          int(r["Game"])   if not pd.isna(r["Game"])   else None,
            "set":           int(r["Set"])    if not pd.isna(r["Set"])    else None,
            "shotNumber":    int(r["Shot"])   if not pd.isna(r["Shot"])   else None,
            "hit_x":         round(float(r["hit_x"]),    3) if not pd.isna(r["hit_x"])    else None,
            "hit_y":         round(float(r["hit_y"]),    3) if not pd.isna(r["hit_y"])    else None,
            "bounce_x":      round(float(r["bounce_x"]), 3) if not pd.isna(r["bounce_x"]) else None,
            "bounce_y":      round(float(r["bounce_y"]), 3) if not pd.isna(r["bounce_y"]) else None,
            "opp_hit_x":     round(float(r["opp_hit_x"]), 3) if "opp_hit_x" in r and not pd.isna(r.get("opp_hit_x", np.nan)) else None,
            "opp_hit_y":     round(float(r["opp_hit_y"]), 3) if "opp_hit_y" in r and not pd.isna(r.get("opp_hit_y", np.nan)) else None,
            "hitSideZone":   r["hit_side_zone"],
            "ballDirection": r["ball_direction"],
            "depthLabel":    r["depth_label"],
            "outcome":       r["miss_quality"],
            "result":        str(r["Result"]).strip(),
            "vec_x":         round(float(r["vec_x"]),         3) if not pd.isna(r["vec_x"])         else None,
            "vec_y":         round(float(r["vec_y"]),         3) if not pd.isna(r["vec_y"])         else None,
            "vec_angle_deg": round(float(r["vec_angle_deg"]), 2) if not pd.isna(r["vec_angle_deg"]) else None,
            "x_deviation":   round(float(r["x_deviation"]),   3) if not pd.isna(r["x_deviation"])   else None,
            "sourceFile":    str(r["__source_file__"])
        }
        records.append(rec)
    return records
 
 
def build_summary_json(df, records, threshold):
    total = len(df)
    if total == 0:
        return {}
    in_s  = df[df["Result"].str.strip().str.lower() == "in"]
    made  = df[df["miss_quality"].isin(["Won", "Neutral"])]
    won   = df[df["miss_quality"] == "Won"]
 
    return {
        "playerName":     player_name,
        "totalShots":     total,
        "shotsMadePct":   round(len(made) / total * 100, 1),
        "pointWinRate":   round(len(won) / len(made) * 100, 1) if len(made) > 0 else 0,
        "outcomes":       {k: {"count": int(v), "pct": round(v/total*100,1)}
                           for k, v in df["miss_quality"].value_counts().items()},
        "depthBreakdown": {k: {"count": int(v), "pct": round(v/len(in_s)*100,1)}
                           for k, v in in_s["depth_label"].value_counts().items()} if len(in_s) > 0 else {},
        "directionBreakdown": {k: {"count": int(v), "pct": round(v/total*100,1)}
                                for k, v in df["ball_direction"].value_counts().items()},
        "hitLocation": {
            "mean_x":    round(float(df["hit_x"].mean()), 2),
            "mean_y":    round(float(df["hit_y"].mean()), 2),
            "median_x":  round(float(df["hit_x"].median()), 2),
            "threshold_x": round(float(threshold), 2)
        },
        "shots": records
    }
 
 
all_records = build_records(io_fh)
bh_records  = build_records(io_fh_bh)
 
with open(f"data/insideout_fh_all.json", "w") as f:
    json.dump(build_summary_json(io_fh, all_records, threshold_x), f, indent=4)
 
with open(f"data/insideout_fh_bh_corner.json", "w") as f:
    json.dump(build_summary_json(io_fh_bh, bh_records, threshold_x), f, indent=4)
 
print(f"\nSaved: data/insideout_fh_all.json")
print(f"Saved: data/insideout_fh_bh_corner.json")


Saved: data/insideout_fh_all.json
Saved: data/insideout_fh_bh_corner.json


In [31]:
export_cols = [
    "Point", "Game", "Set", "Shot", "__source_file__",
    "hit_x", "hit_y", "bounce_x", "bounce_y",
    "hit_side_zone", "ball_direction", "depth_label",
    "miss_quality", "outcome", "Result",
    "vec_x", "vec_y", "vec_angle_deg", "x_deviation",
    "Point Winner"
]
if "opp_hit_x" in io_fh.columns:
    export_cols += ["opp_hit_x", "opp_hit_y"]
export_cols = [c for c in export_cols if c in io_fh.columns]
 
io_fh[export_cols].to_csv(f"data/{player_name}_insideout_fh.csv", index=False)
print(f"Saved: data/{player_name}_insideout_fh.csv")
 
# Summary CSV
summary_rows = []
for label, df_s in [("All IO-FH", io_fh), ("Backhand Corner", io_fh_bh), ("Forehand Corner", io_fh_fh)]:
    if len(df_s) == 0:
        continue
    total = len(df_s)
    in_s  = df_s[df_s["Result"].str.strip().str.lower() == "in"]
    made  = df_s[df_s["miss_quality"].isin(["Won", "Neutral"])]
    won   = df_s[df_s["miss_quality"] == "Won"]
    summary_rows.append({
        "section":             label,
        "total_shots":         total,
        "shots_made_pct":      f"{round(len(made)/total*100,1)}%",
        "point_win_rate":      f"{round(len(won)/len(made)*100,1)}%" if len(made) > 0 else "0%",
        "offensive_deep_pct":  f"{round(len(in_s[in_s['depth_label']=='Offensive'])/len(in_s)*100,1)}%" if len(in_s) > 0 else "0%",
        "defensive_short_pct": f"{round(len(in_s[in_s['depth_label']=='Defensive'])/len(in_s)*100,1)}%" if len(in_s) > 0 else "0%",
        "good_miss_count":     len(df_s[df_s["miss_quality"] == "Good Miss"]),
        "bad_miss_count":      len(df_s[df_s["miss_quality"] == "Bad Miss"]),
        "cross_court_pct":     f"{round(len(df_s[df_s['ball_direction']=='Cross Court'])/total*100,1)}%",
        "down_the_line_pct":   f"{round(len(df_s[df_s['ball_direction']=='Down the Line'])/total*100,1)}%",
        "mean_hit_x":          round(float(df_s["hit_x"].mean()), 2),
        "mean_hit_y":          round(float(df_s["hit_y"].mean()), 2),
        "threshold_x":         round(float(threshold_x), 2),
    })
 
pd.DataFrame(summary_rows).to_csv(f"data/{player_name}_insideout_fh_summary.csv", index=False)
print(f"Saved: data/{player_name}_insideout_fh_summary.csv")

Saved: data/Stian Klaassen_insideout_fh.csv
Saved: data/Stian Klaassen_insideout_fh_summary.csv


In [32]:
def classify_zone(df):
    x = df['x_coord']
    
    if x >= 0:
        side = 'Forehand'
    elif x < 0:
        side = 'Backhand'
    else:
        side = np.nan
    
    return pd.Series({'side': side})

In [33]:
combined_data_shots

player_shots = combined_data_shots[
    (combined_data_shots['Player'] == player_name) &
    (combined_data_shots['Stroke'] == 'Backhand') &
    (combined_data_shots['Direction'] == 'inside out')
].copy()

player_shots


,Player,Shot,Type,Stroke,Spin,Speed (MPH),Point,Game,Set,Bounce Depth,...,Hit Side,Hit (x),Hit (y),Hit (z),Direction,Result,Favorited,Start Time,Video Time,__source_file__
894,Stian Klaassen,2,first_return,Backhand,Flat,41.978043,3,1,1,deep,...,far,-0.621582,24.677538,1.262695,inside out,In,False,22:54:42,66.550003,StianKlaasen_LSU_10_31_25.xlsx
966,Stian Klaassen,2,first_return,Backhand,Flat,37.007816,16,3,1,short,...,near,1.971680,-0.361795,1.294922,inside out,Net,False,23:03:29,593.330017,StianKlaasen_LSU_10_31_25.xlsx
1003,Stian Klaassen,2,first_return,Backhand,Slice,26.929253,24,5,1,deep,...,far,-0.143799,24.036913,1.343750,inside out,Net,False,23:08:58,922.479980,StianKlaasen_LSU_10_31_25.xlsx
1091,Stian Klaassen,2,none,Backhand,Topspin,45.100567,44,9,1,deep,...,far,-0.317871,25.349413,1.169922,inside out,In,False,23:23:32,1796.459961,StianKlaasen_LSU_10_31_25.xlsx
1137,Stian Klaassen,2,first_return,Backhand,Flat,37.623638,50,9,1,deep,...,far,-0.338135,24.130663,0.976074,inside out,In,False,23:28:55,2119.219971,StianKlaasen_LSU_10_31_25.xlsx
1240,Stian Klaassen,2,first_return,Backhand,Slice,35.014233,62,11,2,deep,...,far,-1.134766,23.990038,1.043945,inside out,In,False,23:42:05,2909.919922,StianKlaasen_LSU_10_31_25.xlsx
1304,Stian Klaassen,1,none,Backhand,Flat,36.553722,70,13,2,short,...,near,1.816406,-1.483743,1.011719,inside out,Net,False,23:49:33,3357.030029,StianKlaasen_LSU_10_31_25.xlsx
1320,Stian Klaassen,2,second_return,Backhand,Flat,53.946384,72,13,2,deep,...,near,0.909668,-3.035013,1.552734,inside out,In,False,23:51:15,3459.750000,StianKlaasen_LSU_10_31_25.xlsx
1445,Stian Klaassen,2,first_return,Backhand,Flat,45.078144,94,17,3,deep,...,near,1.994141,-0.859475,1.381836,inside out,In,False,00:11:47,4691.479980,StianKlaasen_LSU_10_31_25.xlsx
1456,Stian Klaassen,2,first_return,Backhand,Flat,30.982164,96,17,3,short,...,near,1.211914,-0.400796,1.252930,inside out,Net,False,00:13:10,4774.589844,StianKlaasen_LSU_10_31_25.xlsx


In [34]:
print(io_fh[['hit_y', 'Hit Zone', 'Bounce Zone']].head(20))

        hit_y Hit Zone Bounce Zone
0   -1.503274       ad          ad
1   25.880663       ad          ad
2   -0.498392       ad          ad
3   24.661913       ad          ad
4   -1.757669       ad          ad
5   -1.572122       ad          ad
6   25.599413       ad          ad
7   25.380663       ad          ad
8   26.052538       ad          ad
9   26.083788       ad          ad
10  24.599413       ad          ad
11  25.521288       ad          ad
12  24.880663       ad          ad
13  -2.010599       ad          ad
14  25.365038       ad          ad
15  25.177538       ad          ad
16  26.193163       ad          ad
17  25.224413       ad          ad
18  24.818163       ad          ad
19  24.333788       ad          ad
